# Parallel Hull-White Calibration with ParFun + QuantLib

The **Hull-White model** is a short-rate model widely used for interest-rate derivatives pricing.
Calibrating it involves fitting model parameters to a grid of swaption prices using
a tree-based pricing engine.

When multiple yield curves need calibration (e.g. different currencies or scenarios),
each calibration is independent and can run in parallel.

ParFun parallelises this with minimal code changes.

## Program Structure

![Program Structure](Simple_Hull_White_Calibration_Parfun_flowchart.svg)

# Native Python Implementation

In [1]:
import time

import QuantLib as ql

sequential_runtime = None


def calibrate_hw_once(curve_data):
    """Calibrate a Hull-White model for a yield curve.
    curve_data = (curve_id, base_rate)
    Returns (curve_id, calibrated_params)."""
    curve_id, base_rate = curve_data
    today = ql.Date(1, 1, 2025)
    ql.Settings.instance().evaluationDate = today

    tenors_months = [0, 6, 12, 24, 36, 60, 84, 120, 180, 240, 360, 480]
    dates = [today + ql.Period(m, ql.Months) for m in tenors_months]
    rates = [base_rate + i * 0.002 for i in range(len(tenors_months))]

    curve = ql.ZeroCurve(dates, rates, ql.Actual365Fixed())
    curve_handle = ql.YieldTermStructureHandle(curve)
    index = ql.Euribor6M(curve_handle)

    model = ql.HullWhite(curve_handle)
    engine = ql.TreeSwaptionEngine(model, 150)

    helpers = []
    for expiry_years in [1, 2, 3, 5, 7, 10, 15, 20]:
        for swap_tenor in [2, 5, 10, 15]:
            h = ql.SwaptionHelper(
                ql.Period(expiry_years, ql.Years),
                ql.Period(swap_tenor, ql.Years),
                ql.QuoteHandle(ql.SimpleQuote(0.0055)),
                index, index.tenor(), index.dayCounter(), index.dayCounter(),
                curve_handle,
            )
            h.setPricingEngine(engine)
            helpers.append(h)

    model.calibrate(
        helpers, ql.LevenbergMarquardt(),
        ql.EndCriteria(5000, 500, 1e-8, 1e-8, 1e-8),
    )
    return curve_id, list(model.params())


def calibrate_hw(curve_data_list):
    """Calibrate Hull-White for each yield curve."""
    return [calibrate_hw_once(d) for d in curve_data_list]


def main():
    global sequential_runtime

    curves = [
        ("curve_A", 0.010),
        ("curve_B", 0.015),
        ("curve_C", 0.005),
        ("curve_D", 0.020),
        ("curve_E", 0.008),
        ("curve_F", 0.012),
        ("curve_G", 0.018),
        ("curve_H", 0.003),
    ]

    start = time.time()
    results = calibrate_hw(curves)
    sequential_runtime = time.time() - start

    print(f"Sequential: {sequential_runtime:.2f}s")


if __name__ == "__main__":
    main()

IOStream.flush timed out


Sequential: 174.63s


# ParFun Implementation

In [2]:
import time

import QuantLib as ql

import parfun as pf

parallel_runtime = None


def calibrate_hw_once(curve_data):
    """Calibrate a Hull-White model for a yield curve.
    curve_data = (curve_id, base_rate)
    Returns (curve_id, calibrated_params)."""
    curve_id, base_rate = curve_data
    today = ql.Date(1, 1, 2025)
    ql.Settings.instance().evaluationDate = today

    tenors_months = [0, 6, 12, 24, 36, 60, 84, 120, 180, 240, 360, 480]
    dates = [today + ql.Period(m, ql.Months) for m in tenors_months]
    rates = [base_rate + i * 0.002 for i in range(len(tenors_months))]

    curve = ql.ZeroCurve(dates, rates, ql.Actual365Fixed())
    curve_handle = ql.YieldTermStructureHandle(curve)
    index = ql.Euribor6M(curve_handle)

    model = ql.HullWhite(curve_handle)
    engine = ql.TreeSwaptionEngine(model, 150)

    helpers = []
    for expiry_years in [1, 2, 3, 5, 7, 10, 15, 20]:
        for swap_tenor in [2, 5, 10, 15]:
            h = ql.SwaptionHelper(
                ql.Period(expiry_years, ql.Years),
                ql.Period(swap_tenor, ql.Years),
                ql.QuoteHandle(ql.SimpleQuote(0.0055)),
                index, index.tenor(), index.dayCounter(), index.dayCounter(),
                curve_handle,
            )
            h.setPricingEngine(engine)
            helpers.append(h)

    model.calibrate(
        helpers, ql.LevenbergMarquardt(),
        ql.EndCriteria(5000, 500, 1e-8, 1e-8, 1e-8),
    )
    return curve_id, list(model.params())


@pf.parfun(
    split=pf.per_argument(curve_data_list=pf.py_list.by_chunk),
    combine_with=pf.py_list.concat,
    fixed_partition_size=1,
)
def calibrate_hw(curve_data_list):
    """Calibrate Hull-White for each yield curve."""
    return [calibrate_hw_once(d) for d in curve_data_list]


def main():
    global parallel_runtime

    curves = [
        ("curve_A", 0.010),
        ("curve_B", 0.015),
        ("curve_C", 0.005),
        ("curve_D", 0.020),
        ("curve_E", 0.008),
        ("curve_F", 0.012),
        ("curve_G", 0.018),
        ("curve_H", 0.003),
    ]

    with pf.set_parallel_backend_context("scaler_local", n_workers=4):
        start = time.time()
        results = calibrate_hw(curves)
        parallel_runtime = time.time() - start

    print(f"Parallel:   {parallel_runtime:.2f}s")


if __name__ == "__main__":
    main()

[INFO]2026-03-31 11:04:27+0800: logging to ('/dev/stdout',)
[INFO]2026-03-31 11:04:27+0800: ObjectStorageServer: start and listen to tcp://127.0.0.1:56587
[INFO]2026-03-31 11:04:27+0800: ObjectStorageServer: started
[INFO]2026-03-31 11:04:27+0800: logging to ('/dev/stdout',)
[INFO]2026-03-31 11:04:27+0800: use event loop: builtin
[INFO]2026-03-31 11:04:27+0800: ConfigController: scheduler_address = tcp://127.0.0.1:41073
[INFO]2026-03-31 11:04:27+0800: ConfigController: object_storage_address = tcp://127.0.0.1:56587
[INFO]2026-03-31 11:04:27+0800: ConfigController: monitor_address = None
[INFO]2026-03-31 11:04:27+0800: ConfigController: protected = True
[INFO]2026-03-31 11:04:27+0800: ConfigController: max_number_of_tasks_waiting = -1
[INFO]2026-03-31 11:04:27+0800: ConfigController: client_timeout_seconds = 60
[INFO]2026-03-31 11:04:27+0800: ConfigController: worker_timeout_seconds = 60
[INFO]2026-03-31 11:04:27+0800: ConfigController: object_retention_seconds = 60
[INFO]2026-03-31 11:

# Diff: Native vs ParFun

In [3]:
import re
import difflib
from IPython.display import display, HTML

native_code = In[1]
parfun_code = In[2]

differ = difflib.HtmlDiff(wrapcolumn=90)
table = differ.make_table(
    native_code.splitlines(),
    parfun_code.splitlines(),
    fromdesc="Native Python",
    todesc="With ParFun",
    context=False,
)

# Strip unwanted columns: navigation links, line numbers, and nowrap attribute
table = re.sub(r'<td class="diff_next"[^>]*>.*?</td>', '', table)
table = re.sub(r'<th class="diff_next"[^>]*>.*?</th>', '', table)
table = re.sub(r'<td class="diff_header"[^>]*>.*?</td>', '', table)
table = table.replace('colspan="2"', '')
table = table.replace(' nowrap="nowrap"', '')
table = re.sub(r'(\s*<colgroup></colgroup>){6}',
               '\n        <colgroup></colgroup> <colgroup></colgroup>', table)

css = """<style>
    table.diff { font-family: monospace; font-size: 10px; border-collapse: collapse; width: 100%; }
    table.diff td { padding: 2px 8px; white-space: pre-wrap; word-break: break-all; text-align: left !important; }
    table.diff th { padding: 6px 8px; text-align: center; background-color: #f0f0f0; font-size: 14px; }
    td.diff_add { background-color: #e6ffec; }
    td.diff_chg { background-color: #fff8c5; }
    td.diff_sub { background-color: #ffebe9; }
    span.diff_add { background-color: #ccffd8; }
    span.diff_chg { background-color: #fff2a8; }
    span.diff_sub { background-color: #ffc0c0; }
</style>"""

display(HTML(css + table))

Native Python,With ParFun
import time,import time
,
import QuantLib as ql,import QuantLib as ql
,
sequential_runtime = None,import parfun as pf
,
,parallel_runtime = None
,
,
def calibrate_hw_once(curve_data):,def calibrate_hw_once(curve_data):


# Speedup

In [4]:
print(f"Sequential: {sequential_runtime:.2f}s")
print(f"Parallel:   {parallel_runtime:.2f}s")
print(f"Speedup:    {sequential_runtime / parallel_runtime:.2f}x")

Sequential: 174.63s
Parallel:   84.99s
Speedup:    2.05x
